<a href="https://colab.research.google.com/github/joshstructure/D2MPDB/blob/main/Bridge_Geo_2_Beam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bridge geometry — configurable beam layout

Edit **Global Bridge Parameters**, then **Restart kernel and Run all** to refresh derived elevations and every output. Changing a variable alone does not refresh existing widgets.

- `num_beams`: positive integer.
- `beam_spacing`: uniform center-to-center spacing, feet.
- `first_beam_offset = None`: center the group on the PGL, including even beam counts. Set a number to hold Beam 1 at a fixed offset instead.
- `beam_spacings = None`: use uniform spacing. Supply exactly `num_beams - 1` gaps for unequal spacing.
- `beam_elevation_offsets`: inches, scalar for all beams or a list ordered by beam number. A shorter list is padded with zeros and a longer list is trimmed, with a warning. Review the displayed layout table after every change.

The active configuration uses two beams with equal vertical offsets. This edition centralizes beam positions and prevents imported roadway grades from overwriting bridge grades. Existing output sections are retained. The FIB 36 drawing remains an approximate fixed section; the superelevation checkbox remains a preview only. See `Bridge_GeoV2_Review.md` for proposed reorganization and remaining model limitations.

Pier-cap and pedestal geometry is in Section 13. Its unconfirmed dimensions are marked `None`; edit those inputs before using the new elevation results.


## 1. Setup and Initialization
Importing required libraries for data manipulation, continuous mathematics, and plotting.

In [ ]:
# Mount Drive only in Colab; local execution uses the adjacent XML.
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount('/content/drive')


In [ ]:
# @title
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Global Bridge Parameters (Multi-Span)
Here you can input the beam parameters and define the multi-span configuration. Each span can have its own length, camber, and haunch buildup requirements.

In [ ]:
# @title
# ==========================================
# GLOBAL BRIDGE PARAMETERS
# ==========================================
import pandas as pd

# 0. Input Files
from pathlib import Path
roadway_xml_path = str(Path('Geometry Report.xml') if Path('Geometry Report.xml').exists() else Path('/content/drive/MyDrive/ColabData/Geometry Report.xml'))  # Updated to point to Google Drive

# 1. Vertical Curve Parameters
vpc_sta = 1593.22             # (ft) Point of Vertical Curve Station
vpc_elev = 47.661             # (ft) Point of Vertical Curve Elevation
vpt_sta = 1805.47             # (ft) Point of Vertical Tangent Station
vpt_elev = 47.673             # (ft) True Point of Vertical Tangent Elevation (0.006 ft higher than VPC)

# Calculate grades to peak exactly at pier and balance near 0.005 magnitude
vpi_sta_target = 1700.57      # (ft) Target peak station (Pier)
L1 = vpi_sta_target - vpc_sta
L2 = vpt_sta - vpi_sta_target

# Balance the error by averaging the theoretical VPI elevations from exactly 0.5% slopes
vpi_elev_from_vpc = vpc_elev + 0.005000 * L1
vpi_elev_from_vpt = vpt_elev + 0.005000 * L2 # using +0.005 to project backwards against a -0.005 slope
vpi_elev_target = (vpi_elev_from_vpc + vpi_elev_from_vpt) / 2

# Derived balanced grades
g1 = (vpi_elev_target - vpc_elev) / L1
g2 = (vpt_elev - vpi_elev_target) / L2

is_brokeback_global = True    # (bool) Toggle for brokeback condition

# 2. Bridge & Beam Cross-Section
num_beams = 2                 # (count) Number of beams
beam_spacing = 4.583         # (ft) Distance between beam centerlines
first_beam_offset = None   # None centers the beam group on PGL; a number anchors Beam 1 (ft)
beam_spacings = None        # Optional unequal gaps in ft, e.g. [4.5, 5.0]; overrides beam_spacing
deck_thickness = 0.5          # (ft) Deck thickness
beam_depth = 3                # (ft) Depth of the beam
flange_width = 48.0           # (inches) Width of the beam top flange
beam_elevation_offsets = 0.0 # (inches) Same vertical offset for both beams; mirrored haunches

# 3. Deck Cross-Slope & Superelevation Transition
is_crowned = True             # (bool) True for crowned deck, False for uniform cross-slope
normal_crown_slope = -0.02    # (ft/ft) Normal cross slope (e.g. -2%)
eval_start = vpc_sta          # (ft) Transition evaluation start station
eval_end = vpt_sta            # (ft) Transition evaluation end station
sta_start = vpc_sta + 10.0    # (ft) Transition start station
sta_end = vpc_sta + 50.0      # (ft) Transition end station
slope_start = -0.02           # (ft/ft) Slope at start of transition
slope_end = 0.04              # (ft/ft) Slope at end of transition

# 4. Multi-Span Configuration & Beam Elevations
haunch = 0.5                  # (inches) Minimum buildup/haunch

spans_config = [
    {'span_id': 1, 'start_sta': 1593.22, 'end_sta': 1700.57, 'camber_max': 1.0},
    {'span_id': 2, 'start_sta': 1700.57, 'end_sta': 1805.47, 'camber_max': 1.0}
]

# Resolve the transverse layout once; all outputs use these arrays.
def resolve_beam_layout(count, spacing, first_offset, elevation_offsets, spacings=None):
    import warnings
    if isinstance(count, (bool, np.bool_)) or not isinstance(count, (int, np.integer)) or count < 1:
        raise ValueError("num_beams must be a positive integer.")
    if spacings is None:
        if not np.isscalar(spacing) or not np.isfinite(spacing) or spacing <= 0:
            raise ValueError("beam_spacing must be a finite positive distance in feet.")
        gaps = np.full(count - 1, float(spacing))
    else:
        gaps = np.asarray(spacings, dtype=float)
        if gaps.shape != (count - 1,) or not np.all(np.isfinite(gaps)) or np.any(gaps <= 0):
            raise ValueError("beam_spacings must contain num_beams - 1 positive finite distances.")
    offsets = np.r_[0.0, np.cumsum(gaps)]
    if first_offset is None:
        offsets -= offsets[-1] / 2.0
    elif np.isscalar(first_offset) and np.isfinite(first_offset):
        offsets += first_offset
    else:
        raise ValueError("first_beam_offset must be None or a finite offset in feet.")
    raises = np.asarray(elevation_offsets, dtype=float)
    if raises.ndim == 0:
        raises = np.full(count, float(raises))
    elif raises.ndim != 1:
        raise ValueError("beam_elevation_offsets must be a scalar or a list in inches.")
    elif len(raises) != count:
        warnings.warn("Resizing beam_elevation_offsets by beam number: added beams receive 0 in; removed beams are discarded. Review the layout table.")
        raises = np.r_[raises[:count], np.zeros(max(0, count - len(raises)))]
    if not np.all(np.isfinite(raises)):
        raise ValueError("Beam elevation offsets must be finite.")
    return offsets, raises.tolist()

beam_offsets_ft, beam_elevation_offsets = resolve_beam_layout(
    num_beams, beam_spacing, first_beam_offset, beam_elevation_offsets, beam_spacings)
if num_beams > 1 and np.any(np.diff(beam_offsets_ft) < flange_width / 12.0):
    raise ValueError("Beam spacing is smaller than the top flange width; flanges overlap.")
display(pd.DataFrame({'Beam': np.arange(1, num_beams + 1),
                      'Offset from PGL (ft)': beam_offsets_ft,
                      'Vertical offset (in)': beam_elevation_offsets}))

# --- Dynamic End Elevation Calculation ---
def _calc_pgl_for_elev(sta):
    # Mathematical intersection of entering and exiting tangent lines
    vpi_sta_calc = (vpt_elev - vpc_elev + g1 * vpc_sta - g2 * vpt_sta) / (g1 - g2)
    vpi_elev_calc = vpc_elev + g1 * (vpi_sta_calc - vpc_sta)

    if is_brokeback_global:
        if sta <= vpi_sta_calc:
            return vpc_elev + g1 * (sta - vpc_sta)
        else:
            return vpi_elev_calc + g2 * (sta - vpi_sta_calc)
    else:
        # Standard parabola
        L_vc = vpt_sta - vpc_sta
        r = (g2 - g1) / L_vc
        x_vc = sta - vpc_sta
        return vpc_elev + g1 * x_vc + 0.5 * r * (x_vc ** 2)

for i, span in enumerate(spans_config):
    pgl_start = _calc_pgl_for_elev(span['start_sta'])
    pgl_end = _calc_pgl_for_elev(span['end_sta'])

    if i == 0:
        worst_pinch = 0
        for b in range(num_beams):
            offset = beam_offsets_ft[b]
            beam_raise = (beam_elevation_offsets[b] / 12.0) if b < len(beam_elevation_offsets) else 0.0
            for p_off in [offset, offset - (flange_width / 24.0), offset + (flange_width / 24.0)]:
                slope = -normal_crown_slope if (is_crowned and p_off < 0) else normal_crown_slope
                deck_drop = -(p_off * slope)
                pinch = deck_drop + beam_raise
                if pinch > worst_pinch:
                    worst_pinch = pinch

        span['beam_bottom_start_elev'] = pgl_start - deck_thickness - (haunch / 12.0) - (span['camber_max'] / 12.0) - beam_depth - worst_pinch
    else:
        span['beam_bottom_start_elev'] = spans_config[i-1]['beam_bottom_end_elev']

    span['beam_bottom_end_elev'] = span['beam_bottom_start_elev'] + (pgl_end - pgl_start)

spans_df = pd.DataFrame(spans_config)
spans_df['length'] = spans_df['end_sta'] - spans_df['start_sta']

bridge_start_sta = spans_df['start_sta'].min()
bridge_end_sta = spans_df['end_sta'].max()
bridge_length = bridge_end_sta - bridge_start_sta

print(f"Global parameters updated.")
print(f"Target VPI Elev: {vpi_elev_target:.4f}")
print(f"Calculated g1: {g1:.6f} | Calculated g2: {g2:.6f}")
display(spans_df)


In [ ]:
# @title
# This duplicate parameters cell has been cleared to prevent conflicts.
# Please use the global parameters cell above.

In [ ]:
# File discovery is optional; the configured path is used below.
print(f'Roadway XML: {roadway_xml_path}')


In [ ]:
# @title
# --- Crown & Offset-based Slope Logic ---
# Note: is_crowned and normal_crown_slope are now managed in Global Parameters (Section 2)

def get_effective_cross_slope(offset, sta=None):
    """
    Returns the appropriate cross-slope based on the beam offset.
    If the deck is crowned, beams on the left (negative offset)
    need a positive slope multiplier so the elevation drops.
    """
    if is_crowned:
        # Left side drops (negative offset * positive slope = lower elevation)
        # Right side drops (positive offset * negative slope = lower elevation)
        return -normal_crown_slope if offset < 0 else normal_crown_slope
    else:
        # Fallback to a single uniform slope (superelevation)
        return normal_crown_slope

print("Offset-based cross-slope logic defined.")

In [ ]:
# @title
import numpy as np

def get_pgl_elevation(sta):
    """
    Calculates the Profile Grade Line (PGL) elevation for a given station.
    Supports both scalar (float) and numpy array inputs seamlessly.
    Uses true geometric intersection for brokeback curves.
    """
    is_bb = globals().get('is_brokeback_global', False)

    # Retrieve globals directly for robust evaluation
    local_vpc_sta = globals().get('vpc_sta', 1593.22)
    local_vpc_elev = globals().get('vpc_elev', 47.667)
    local_g1 = globals().get('g1', 0.005058)
    local_g2 = globals().get('g2', -0.004942)
    local_vpt_sta = globals().get('vpt_sta', 1805.47)
    local_vpt_elev = globals().get('vpt_elev', 47.673)

    if is_bb:
        # Mathematical intersection of entering and exiting tangents
        vpi_sta_calc = (local_vpt_elev - local_vpc_elev + local_g1 * local_vpc_sta - local_g2 * local_vpt_sta) / (local_g1 - local_g2)
        vpi_elev_calc = local_vpc_elev + local_g1 * (vpi_sta_calc - local_vpc_sta)

        return np.where(sta <= vpi_sta_calc,
                        local_vpc_elev + local_g1 * (sta - local_vpc_sta),
                        vpi_elev_calc + local_g2 * (sta - vpi_sta_calc))
    else:
        L_vc = local_vpt_sta - local_vpc_sta
        r = (local_g2 - local_g1) / L_vc
        x_vc = sta - local_vpc_sta
        return local_vpc_elev + local_g1 * x_vc + 0.5 * r * (x_vc ** 2)

print("Global PGL elevation helper updated to use true tangent intersection math.")


## 3. Continuous Mathematical Model: Vertical Curve
Here we define the continuous parabolic equation for the vertical curve based on the standard highway engineering formula. We transition from discrete spreadsheet rows to a continuous NumPy array.

In [ ]:
# @title
def calculate_vertical_curve(vpc_sta, vpc_elev, g1, g2, vpt_sta, num_points=500, is_brokeback=False, vpi_sta=None):
    """
    Calculates the continuous vertical curve elevations.
    Uses the global vpt_elev to calculate the true geometric intersection if brokeback is True.
    """
    L = vpt_sta - vpc_sta
    x = np.linspace(0, L, num_points)
    stations = vpc_sta + x

    local_vpt_elev = globals().get('vpt_elev', vpc_elev + (g1 * L/2) + (g2 * L/2))

    if is_brokeback:
        # True intersection math
        vpi_sta_calc = (local_vpt_elev - vpc_elev + g1 * vpc_sta - g2 * vpt_sta) / (g1 - g2)
        vpi_elev_calc = vpc_elev + g1 * (vpi_sta_calc - vpc_sta)
        vpi_x = vpi_sta_calc - vpc_sta

        y = np.where(x <= vpi_x,
                     vpc_elev + g1 * x,
                     vpi_elev_calc + g2 * (x - vpi_x))
    else:
        if vpi_sta is not None:
            vpi_x = vpi_sta - vpc_sta
        else:
            # If symmetric
            vpi_x = L / 2

        if vpi_sta is None or np.isclose(vpi_x, L / 2):
            r = (g2 - g1) / L
            y = vpc_elev + g1 * x + 0.5 * r * (x ** 2)
        else:
            # Asymmetric parabolic equation
            L1 = vpi_x
            L2 = L - vpi_x
            y = np.where(
                x <= L1,
                vpc_elev + g1 * x + ((g2 - g1) * L2 / (2 * L * L1)) * (x ** 2),
                (vpc_elev + g1 * L1) + g2 * (x - L1) + ((g2 - g1) * L1 / (2 * L * L2)) * ((L - x) ** 2)
            )

    return stations, y


## 4. Interactive Vertical Curve Plot
Using the parameters we observed in your Excel file, we set up interactive sliders. You can change these inputs to immediately update the continuous mathematical plot.

In [ ]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output

def plot_vertical_curve_static(vpc_sta, vpc_elev, g1, g2, vpt_sta, inspect_sta=None, is_brokeback=False):
    stations, elevations = calculate_vertical_curve(vpc_sta, vpc_elev, g1, g2, vpt_sta, is_brokeback=is_brokeback)

    plt.figure(figsize=(12, 6))
    plt.plot(stations, elevations, label='Continuous Profile', color='darkblue', linewidth=2)
    plt.scatter([vpc_sta, vpt_sta], [elevations[0], elevations[-1]], color='red', zorder=5)

    # Calculate VPI for tangent lines
    L = vpt_sta - vpc_sta
    local_vpt_elev = globals().get('vpt_elev', vpc_elev + (g1 * L/2) + (g2 * L/2))
    vpi_sta_calc = (local_vpt_elev - vpc_elev + g1 * vpc_sta - g2 * vpt_sta) / (g1 - g2)
    vpi_elev_calc = vpc_elev + g1 * (vpi_sta_calc - vpc_sta)

    # Draw Tangents
    plt.plot([vpc_sta, vpi_sta_calc, vpt_sta], [vpc_elev, vpi_elev_calc, local_vpt_elev],
             color='gray', linestyle='--', linewidth=1.5, label='Tangents', zorder=3)

    # Annotate g1 and g2 separately next to their tangent lines
    plt.text((vpc_sta + vpi_sta_calc)/2, (vpc_elev + vpi_elev_calc)/2 + 0.05, f"g1: {g1*100:.3f}%",
             ha='center', va='bottom', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.9), zorder=10)

    plt.text((vpi_sta_calc + vpt_sta)/2, (vpi_elev_calc + local_vpt_elev)/2 + 0.05, f"g2: {g2*100:.3f}%",
             ha='center', va='bottom', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.9), zorder=10)

    if inspect_sta is not None:
        dx = inspect_sta - vpc_sta

        if is_brokeback:
            if inspect_sta <= vpi_sta_calc:
                elev = vpc_elev + g1 * (inspect_sta - vpc_sta)
            else:
                elev = vpi_elev_calc + g2 * (inspect_sta - vpi_sta_calc)
        else:
            r = (g2 - g1) / L
            elev = vpc_elev + g1 * dx + 0.5 * r * (dx ** 2)

        plt.axvline(inspect_sta, color='purple', linestyle='--', label=f'Sta: {inspect_sta:.2f}')
        plt.scatter(inspect_sta, elev, color='purple', zorder=6)
        plt.title(f'Continuous Profile | Elev: {elev:.4f} ft | Brokeback: {is_brokeback}')
    else:
        plt.title(f'Continuous Profile | Brokeback: {is_brokeback}')

    plt.xlabel('Station')
    plt.ylabel('Elevation (ft)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='lower center')
    plt.tight_layout()
    plt.show()

print("--- Vertical Curve Model ---")
slider = widgets.FloatSlider(value=vpc_sta, min=vpc_sta, max=vpt_sta, step=0.01, description='Station:', layout=widgets.Layout(width='50%'))
num = widgets.FloatText(value=vpc_sta, description='Exact:', step=0.01, layout=widgets.Layout(width='20%'))
# Use the global parameter to set the default state of the checkbox safely
default_brokeback = globals().get('is_brokeback_global', False)
brokeback_toggle = widgets.Checkbox(value=default_brokeback, description='Brokeback (Intersecting Tangents)', layout=widgets.Layout(width='30%'))

widgets.jslink((slider, 'value'), (num, 'value'))
out = widgets.Output()

def _update(change):
    with out:
        clear_output(wait=True)
        plot_vertical_curve_static(vpc_sta, vpc_elev, g1, g2, vpt_sta, inspect_sta=slider.value, is_brokeback=brokeback_toggle.value)

slider.observe(_update, names='value')
brokeback_toggle.observe(_update, names='value')
display(widgets.HBox([slider, num, brokeback_toggle]), out)
_update(None)

## 5. Continuous Mathematical Model: Deck Slopes (Superelevation)
The Excel file evaluates deck slopes at discrete points. Here, we model the cross-slope as a continuous function. Typically, a deck transitions from a normal cross-slope (e.g., -2%) to a full superelevation (e.g., +4% or -4%) linearly over a specific station range.

We will define a piecewise continuous function to calculate and visualize the deck slope dynamically.

In [ ]:
# @title
def plot_deck_slopes(eval_start, eval_end, sta_start, sta_end, slope_start, slope_end, inspect_sta=None, enable_super=False):
    # Ensure we use an existing fallback if calculate_continuous_deck_slope wasn't defined
    stations = np.linspace(eval_start, eval_end, 500)

    if enable_super:
        if 'calculate_continuous_deck_slope' in globals():
            slopes = calculate_continuous_deck_slope(stations, sta_start, sta_end, slope_start, slope_end)
        else:
            slopes = np.interp(stations, [sta_start, sta_end], [slope_start, slope_end])
    else:
        # Default to the constant normal crown slope if transition is disabled
        slopes = np.full_like(stations, normal_crown_slope)

    plt.figure(figsize=(12, 5))
    plt.plot(stations, slopes * 100, label='Continuous Cross-Slope', color='purple', linewidth=2.5)

    if enable_super:
        plt.axvline(sta_start, color='gray', linestyle='--', label='Transition Start')
        plt.axvline(sta_end, color='gray', linestyle=':', label='Transition End')

    plt.axhline(0, color='black', linewidth=1)

    if inspect_sta is not None:
        idx = (np.abs(stations - inspect_sta)).argmin()
        plt.axvline(inspect_sta, color='orange', linestyle='--')
        plt.scatter(inspect_sta, slopes[idx]*100, color='orange', zorder=6)
        plt.title(f'Continuous Deck Cross-Slope | Sta {inspect_sta:.2f} | Slope: {slopes[idx]*100:.2f}%')
    else:
        title_text = 'Continuous Deck Cross-Slope Transition (Superelevation)' if enable_super else 'Constant Normal Crown Cross-Slope'
        plt.title(title_text)

    plt.xlabel('Station')
    plt.ylabel('Cross-Slope (%)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

print("--- Deck Slope Transition Model ---")
slider3 = widgets.FloatSlider(value=sta_start, min=eval_start, max=eval_end, step=0.01, description='Station:', layout=widgets.Layout(width='50%'))
num3 = widgets.FloatText(value=sta_start, description='Exact:', step=0.01, layout=widgets.Layout(width='20%'))

# New Checkbox to disable the transition entirely
super_toggle = widgets.Checkbox(value=False, description='Enable Superelevation Transition', layout=widgets.Layout(width='50%'))

widgets.jslink((slider3, 'value'), (num3, 'value'))
out3 = widgets.Output()

def _update3(change):
    with out3:
        clear_output(wait=True)
        plot_deck_slopes(eval_start, eval_end, sta_start, sta_end, slope_start, slope_end,
                         inspect_sta=slider3.value, enable_super=super_toggle.value)

slider3.observe(_update3, names='value')
super_toggle.observe(_update3, names='value')
display(widgets.HBox([slider3, num3]), super_toggle, out3)
_update3(None)


## 6. Top of Deck Elevations for All Beams
To calculate the top of deck elevation along the centerline of each beam, we combine the longitudinal **Vertical Curve** profile with the lateral **Continuous Deck Slopes**.

Formula:
`Top of Deck Elevation(x) = Profile Elevation(x) + (Beam Offset * Cross-Slope(x))`

In [ ]:
# @title
def plot_all_beams_deck_elevations(vpc_sta, vpc_elev, g1, g2, vpt_sta,
                                   num_beams, beam_spacing, first_beam_offset, inspect_sta=None):
    is_bb = globals().get('is_brokeback_global', False)
    stations, profile = calculate_vertical_curve(vpc_sta, vpc_elev, g1, g2, vpt_sta, is_brokeback=is_bb)

    plt.figure(figsize=(14, 7))
    line_styles = ['-', '--', ':', '-.']
    # Use high-contrast colors to prevent blue/green bleed
    colors = ['#1f77b4', '#ff7f0e', '#d62728', '#2ca02c', '#9467bd', '#8c564b']

    for i in range(int(num_beams)):
        offset = beam_offsets_ft[i]
        cross_slope = get_effective_cross_slope(offset)
        beam_deck_elev = profile + (offset * cross_slope)

        # Calculate physical slope away from PGL for display
        display_slope = cross_slope if offset >= 0 else -cross_slope

        # Vary linewidth and style so overlapping lines are distinguishable
        lw = max(1.5, 4.0 - i)
        ls = line_styles[i % len(line_styles)]
        c = colors[i % len(colors)]

        # Explicitly set zorder so thinner lines render on top of thicker lines
        plt.plot(stations, beam_deck_elev,
                 label=f'Beam {i+1} (Offset from PGL: {offset:+.2f} ft | Slope: {display_slope*100:+.2f}%)',
                 linestyle=ls, linewidth=lw, color=c, alpha=0.85, zorder=20 - lw)

        if inspect_sta is not None:
            idx = (np.abs(stations - inspect_sta)).argmin()
            plt.scatter(inspect_sta, beam_deck_elev[idx], color='purple', zorder=25)

    plt.plot(stations, profile, label='Profile Grade Line (PGL)', color='black', linewidth=1.5, linestyle='-', zorder=30)

    if inspect_sta is not None:
        plt.axvline(inspect_sta, color='purple', linestyle='--', zorder=0)
        plt.title(f'Continuous Top of Deck Elevations | Sta {inspect_sta:.2f}')
    else:
        plt.title('Continuous Top of Deck Elevations Along Beam Centerlines')

    plt.xlabel('Station')
    plt.ylabel('Elevation (ft)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

print("--- Top of Deck Elevations Model ---")
slider4 = widgets.FloatSlider(value=vpc_sta, min=vpc_sta, max=vpt_sta, step=0.01, description='Station:', layout=widgets.Layout(width='50%'))
num4 = widgets.FloatText(value=vpc_sta, description='Exact:', step=0.01, layout=widgets.Layout(width='20%'))
widgets.jslink((slider4, 'value'), (num4, 'value'))
out4 = widgets.Output()

def _update4(change):
    with out4:
        clear_output(wait=True)
        plot_all_beams_deck_elevations(vpc_sta, vpc_elev, g1, g2, vpt_sta,
                                       num_beams, beam_spacing, first_beam_offset, inspect_sta=slider4.value)

slider4.observe(_update4, names='value')
display(widgets.HBox([slider4, num4]), out4)
_update4(None)


In [ ]:
# @title
import os
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

# The roadway_xml_path is now defined in Section 2 (Global Parameters)
if not os.path.exists(roadway_xml_path):
    print(f"Error: The file '{roadway_xml_path}' was not found.")
    print("Please upload your XML file and verify the path in the Global Parameters.")
else:
    tree = ET.parse(roadway_xml_path)
    root = tree.getroot()

    sta_list = []
    elev_list = []

    def get_point(pt_elem):
        sta = float(pt_elem.find('station').attrib['externalStation'])
        elev = float(pt_elem.attrib['elevation'])
        return sta, elev

    for va in root.iter('VerticalAlignment'):
        for elements in va.iter('VerticalElements'):
            for child in elements:
                if child.tag == 'VerticalLine':
                    start_pt = child.find('VerticalStart')
                    end_pt = child.find('VerticalEnd')
                    if start_pt is not None and end_pt is not None:
                        s_sta, s_elev = get_point(start_pt)
                        e_sta, e_elev = get_point(end_pt)
                        stas = np.linspace(s_sta, e_sta, max(2, int(e_sta - s_sta)))
                        elevs = np.linspace(s_elev, e_elev, len(stas))
                        sta_list.extend(stas)
                        elev_list.extend(elevs)
                elif child.tag == 'VerticalParabola':
                    start_pt = child.find('VerticalStart')
                    end_pt = child.find('VerticalEnd')
                    pi_pt = child.find('VerticalPI')
                    if start_pt is not None and end_pt is not None and pi_pt is not None:
                        s_sta, s_elev = get_point(start_pt)
                        e_sta, e_elev = get_point(end_pt)
                        pi_sta, pi_elev = get_point(pi_pt)
                        L = e_sta - s_sta
                        road_g1 = (pi_elev - s_elev) / (pi_sta - s_sta) if pi_sta != s_sta else 0
                        road_g2 = (e_elev - pi_elev) / (e_sta - pi_sta) if e_sta != pi_sta else 0
                        r = (road_g2 - road_g1) / L if L != 0 else 0
                        stas = np.linspace(s_sta, e_sta, max(10, int(L * 2)))
                        x = stas - s_sta
                        elevs = s_elev + road_g1 * x + 0.5 * r * (x ** 2)
                        sta_list.extend(stas)
                        elev_list.extend(elevs)

    roadway_profile = pd.DataFrame({'Station': sta_list, 'Elevation': elev_list})
    roadway_profile = roadway_profile.drop_duplicates(subset=['Station']).sort_values(by='Station')

    # Map roadway stations to bridge stations (0.00 Roadway = bridge_start_sta)
    roadway_profile['Station'] = roadway_profile['Station'] + bridge_start_sta

    display(roadway_profile.head(10))
    dynamic_roadway_interp = interp1d(roadway_profile['Station'], roadway_profile['Elevation'],
                                      kind='linear', fill_value='extrapolate')

    print(f"\nHigh-resolution roadway model created from {roadway_xml_path} starting at {bridge_start_sta:.2f}!")

In [ ]:
# @title
import matplotlib.pyplot as plt

# Plot the newly extracted high-resolution roadway profile
plt.figure(figsize=(14, 5))

# Base Roadway Elevation
plt.plot(roadway_profile['Station'], roadway_profile['Elevation'],
         color='gray', marker='.', markersize=4, linestyle='-', linewidth=1.5, alpha=0.8, label='Roadway')

# Minimum Vertical Clearance Envelope (Roadway + 17ft)
plt.plot(roadway_profile['Station'], roadway_profile['Elevation'] + 17,
         color='red', linestyle='--', linewidth=2, label='Minimum Vertical Clearance (17 ft)')

plt.title('Roadway Profile & Minimum Vertical Clearance Envelope')
plt.xlabel('Bridge Station')
plt.ylabel('Elevation (ft)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Total number of extracted points: {len(roadway_profile)}")
print(f"Profile covers bridge stations from {roadway_profile['Station'].min():.2f} to {roadway_profile['Station'].max():.2f}")

## 7. Roadway Profile XML Parsing & Interactive Data Inspector
The cells below first extract the high-resolution roadway profile from the XML report and map it to the bridge stations.

Then, they launch the **Master Interactive Data Inspector**. Use the slider or type an exact station into the input box to examine precise analytical values at any location along the span. The visualization will update dynamically to show your location on the curve.

In [ ]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt

station_slider = widgets.FloatSlider(value=bridge_start_sta + (bridge_length/2), min=bridge_start_sta, max=bridge_start_sta + bridge_length, step=0.01, description='Station:', layout=widgets.Layout(width='60%'))
station_input = widgets.FloatText(value=bridge_start_sta + (bridge_length/2), description='Exact Sta:', step=0.01, layout=widgets.Layout(width='25%'))
widgets.jslink((station_slider, 'value'), (station_input, 'value'))
out_inspector = widgets.Output()

def update_inspector(change):
    with out_inspector:
        clear_output(wait=True)
        sta = station_slider.value

        # --- Use new DRY Helper for scalar PGL ---
        pgl_elev = get_pgl_elevation(sta)

        current_span = spans_df[(spans_df['start_sta'] <= sta) & ((spans_df['start_sta'] + spans_df['length']) >= sta)].iloc[0] if not spans_df[(spans_df['start_sta'] <= sta) & ((spans_df['start_sta'] + spans_df['length']) >= sta)].empty else spans_df.iloc[-1]

        stations_arr = np.linspace(bridge_start_sta, bridge_end_sta, 500)

        # --- Use new DRY Helper for continuous array PGL ---
        pgl_arr = get_pgl_elevation(stations_arr)

        camber_arr = np.zeros_like(stations_arr)
        base_bot_arr = np.zeros_like(stations_arr)
        for _, row in spans_df.iterrows():
            mask = (stations_arr >= row['start_sta']) & (stations_arr <= row['end_sta'])
            local_x = stations_arr[mask] - row['start_sta']
            camber_arr[mask] = 4 * (row['camber_max'] / 12) * (local_x * (row['length'] - local_x)) / (row['length'] ** 2)
            base_bot_arr[mask] = row['beam_bottom_start_elev'] + (row['beam_bottom_end_elev'] - row['beam_bottom_start_elev']) * (local_x / row['length'])

        roadway_arr = dynamic_roadway_interp(stations_arr) if 'dynamic_roadway_interp' in globals() else np.full_like(stations_arr, 43.04)

        # --- Exact Calculations for Readout ---
        min_bot_arr = np.full_like(stations_arr, np.inf)
        for i in range(num_beams):
            beam_v_offset = (beam_elevation_offsets[i] / 12.0) if i < len(beam_elevation_offsets) else 0.0
            actual_bot_arr_temp = base_bot_arr + camber_arr + beam_v_offset
            min_bot_arr = np.minimum(min_bot_arr, actual_bot_arr_temp)

        clearance_arr = min_bot_arr - roadway_arr
        worst_clearance_idx = np.argmin(clearance_arr)
        worst_clearance = clearance_arr[worst_clearance_idx]
        worst_sta = stations_arr[worst_clearance_idx]

        x_span = sta - current_span['start_sta']
        s_len, s_camber = current_span['length'], current_span['camber_max']
        s_bot_start, s_bot_end = current_span['beam_bottom_start_elev'], current_span['beam_bottom_end_elev']
        camber_ft = 4 * (s_camber / 12) * (x_span * (s_len - x_span)) / (s_len ** 2)
        base_bot_ft = s_bot_start + (s_bot_end - s_bot_start) * (x_span / s_len)
        lowest_beam_bot_sta = min([base_bot_ft + camber_ft + ((beam_elevation_offsets[i] / 12.0) if i < len(beam_elevation_offsets) else 0.0) for i in range(num_beams)])

        roadway_elev = float(dynamic_roadway_interp(sta)) if 'dynamic_roadway_interp' in globals() else 43.04
        clearance = lowest_beam_bot_sta - roadway_elev

        print(f"--- Data Inspector Readout @ Station {sta:.2f} ---")
        print(f"Profile Grade Line (PGL): {pgl_elev:.3f} ft")
        print(f"Roadway Elevation:        {roadway_elev:.3f} ft")
        print(f"Lowest Beam Bottom:       {lowest_beam_bot_sta:.3f} ft")
        print(f"Vertical Clearance:       {clearance:.3f} ft")
        print(f"Worst Case Clearance:     {worst_clearance:.3f} ft @ Sta {worst_sta:.2f}")
        if clearance < 17.0 or worst_clearance < 17.0:
            print("⚠️ WARNING: Minimum Vertical Clearance is below 17.0 ft!")
        print("-" * 60)

        # Modified figsize for taller plots
        fig1, ax1 = plt.subplots(figsize=(14, 7))
        ax1.plot(stations_arr, pgl_arr, label='PGL (Deck Profile)', color='black', linestyle='--', linewidth=1.5)
        ax1.plot(stations_arr, roadway_arr, color='gray', alpha=0.5, label='Roadway')
        ax1.plot(stations_arr, roadway_arr + 17, color='red', linestyle='--', label='17ft Clearance Envelope')

        fig2, ax2 = plt.subplots(figsize=(14, 8)) # Adjusted height here
        cmap = plt.get_cmap('Set1')
        flange_half_ft = (flange_width / 12) / 2

        for i in reversed(range(num_beams)):
            offset = beam_offsets_ft[i]
            c = cmap(i % cmap.N)
            lw = 1.5 + 0.5 * (i % 4)
            beam_v_offset = (beam_elevation_offsets[i] / 12.0) if i < len(beam_elevation_offsets) else 0.0

            actual_bot_arr = base_bot_arr + camber_arr + beam_v_offset
            actual_tob_arr = actual_bot_arr + beam_depth
            ax1.plot(stations_arr, actual_bot_arr, color=c, label=f'B{i+1} Bottom', linewidth=lw)

            pts = {'CL': (offset, '-'), 'Left': (offset - flange_half_ft, '--'), 'Right': (offset + flange_half_ft, ':')}
            for p_name, (p_off, ls) in pts.items():
                tod_local_arr = pgl_arr + (p_off * get_effective_cross_slope(p_off))
                local_h_arr = tod_local_arr - deck_thickness - actual_tob_arr
                ax2.plot(stations_arr, local_h_arr * 12, label=f'B{i+1} {p_name}', color=c, linestyle=ls, linewidth=lw)

        # Add purple vertical indicator line to both plots for visual tracking
        ax1.axvline(sta, color='purple', alpha=0.3)
        ax2.axvline(sta, color='purple', alpha=0.3)

        ax1.set_title('Bridge Elevation & Vertical Clearance')
        ax1.set_ylabel('Elevation (ft)')
        ax1.legend(loc='upper left', bbox_to_anchor=(1,1))

        ax2.axhline(haunch, color='red', linewidth=2, label=f'Min Haunch ({haunch} in)')
        ax2.set_title('Calculated Haunch Depth Across Flange Width (in)')
        ax2.set_ylabel('Haunch Thickness (in)')
        ax2.set_xlabel('Station')
        ax2.legend(loc='upper left', bbox_to_anchor=(1,1))

        plt.tight_layout()
        plt.show()

station_slider.observe(update_inspector, names='value')
update_inspector(None)
display(widgets.HBox([station_slider, station_input]), out_inspector)


## 8. Mathematical Verification & Component Breakdown
Use this tool as a transparent double-check. Select a specific station to see exactly how the global properties and beam components combine to produce the final Top of Deck, Structure Depth, Beam Bottom, and Clearance values. The exact equations are traced for verification.

In [ ]:
# @title
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

verify_slider = widgets.FloatSlider(value=bridge_start_sta + (bridge_length/2), min=bridge_start_sta, max=bridge_end_sta, step=0.01, description='Station:', layout=widgets.Layout(width='60%'))
verify_input = widgets.FloatText(value=bridge_start_sta + (bridge_length/2), description='Exact Sta:', step=0.01, layout=widgets.Layout(width='25%'))
widgets.jslink((verify_slider, 'value'), (verify_input, 'value'))
out_verify = widgets.Output()

def update_verification(change):
    with out_verify:
        clear_output(wait=True)
        sta = verify_slider.value

        # --- Use new DRY Helper for scalar PGL ---
        pgl_elev = get_pgl_elevation(sta)

        current_span = spans_df[(spans_df['start_sta'] <= sta) & (spans_df['end_sta'] >= sta)].iloc[0]
        x_span = sta - current_span['start_sta']
        s_len, s_camber = current_span['length'], current_span['camber_max']
        s_bot_start, s_bot_end = current_span['beam_bottom_start_elev'], current_span['beam_bottom_end_elev']

        camber_ft = 4 * (s_camber / 12) * (x_span * (s_len - x_span)) / (s_len ** 2)
        base_bot_ft = s_bot_start + (s_bot_end - s_bot_start) * (x_span / s_len)

        breakdown_data = []
        for i in range(num_beams):
            offset = beam_offsets_ft[i]
            beam_v_offset = (beam_elevation_offsets[i] / 12.0) if i < len(beam_elevation_offsets) else 0.0
            actual_bot_ft = base_bot_ft + camber_ft + beam_v_offset
            actual_tob_ft = actual_bot_ft + beam_depth

            for pt in ['Left Edge', 'Centerline', 'Right Edge']:
                p_off = offset - (flange_width/24) if pt == 'Left Edge' else (offset + (flange_width/24) if pt == 'Right Edge' else offset)
                tod_local_ft = pgl_elev + (p_off * get_effective_cross_slope(p_off, sta))
                h_val_in = (tod_local_ft - deck_thickness - actual_tob_ft) * 12

                breakdown_data.append({
                    "Beam": f"Beam {i+1}",
                    "Point": pt,
                    "Top of Deck (ft)": round(tod_local_ft, 3),
                    "Actual Beam Bot (ft)": round(actual_bot_ft, 3),
                    "Actual Beam Top (ft)": round(actual_tob_ft, 3),
                    "Camber (in)": round(camber_ft * 12, 3),
                    "Calculated Haunch (in)": round(h_val_in, 3)
                })

        print(f"--- HAUNCH VERIFICATION (Beam Bottom + {beam_depth}ft Depth Logic) ---")
        display(pd.DataFrame(breakdown_data).style.hide(axis="index"))

verify_slider.observe(update_verification, names='value')
update_verification(None)
display(widgets.HBox([verify_slider, verify_input]), out_verify)


In [ ]:
# @title
def explain_camber_at_station(sta):
    # Find the active span
    span_matches = spans_df[(spans_df['start_sta'] <= sta) & (spans_df['end_sta'] >= sta)]
    if span_matches.empty:
        print('Station is outside of bridge spans.')
        return

    span = span_matches.iloc[0]
    x_span = sta - span['start_sta']
    L = span['length']
    camber_max = span['camber_max'] # in inches

    # Parabolic camber formula: y = 4 * c_max * (x * (L - x)) / L^2
    camber_val = 4 * camber_max * (x_span * (L - x_span)) / (L ** 2)

    print(f'--- Camber Calculation Breakdown for Station {sta:.2f} ---')
    print(f'Span ID: {span["span_id"]}')
    print(f'Max Camber (c_max): {camber_max} in')
    print(f'Span Length (L): {L:.3f} ft')
    print(f'Distance into Span (x): {x_span:.3f} ft')
    print(f'Formula: 4 * {camber_max} * ({x_span:.3f} * ({L:.3f} - {x_span:.3f})) / {L:.3f}^2')
    print(f'Resulting Camber: {camber_val:.4f} inches')

# Check the current slider value
explain_camber_at_station(verify_slider.value)

## 9. Top-of-Deck vs Beam-Bottom Visualization
This chart plots the continuous Top-of-Deck elevations directly alongside the Beam-Bottom elevations. The shaded area represents the entire superstructure depth (deck + haunch - camber + beam depth + offsets).

In [ ]:
# @title
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(14, 7))

# Prepare spatial arrays
x_arr = np.linspace(0, bridge_length, 500)
stations_arr = bridge_start_sta + x_arr

# --- Use new DRY Helper for continuous array PGL ---
pgl_arr = get_pgl_elevation(stations_arr)

# Compute piecewise arrays across spans using Beam Bottom inputs
camber_arr = np.zeros_like(stations_arr)
base_bot_arr = np.zeros_like(stations_arr)

for _, row in spans_df.iterrows():
    mask = (stations_arr >= row['start_sta']) & (stations_arr <= row['end_sta'])
    local_x = stations_arr[mask] - row['start_sta']
    camber_arr[mask] = 4 * (row['camber_max'] / 12) * (local_x * (row['length'] - local_x)) / (row['length'] ** 2)
    base_bot_arr[mask] = row['beam_bottom_start_elev'] + (row['beam_bottom_end_elev'] - row['beam_bottom_start_elev']) * (local_x / row['length'])

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for i in range(num_beams):
    offset = beam_offsets_ft[i]
    c = colors[i % len(colors)]
    slope = get_effective_cross_slope(offset)

    # Top of Deck for this beam
    top_of_deck = pgl_arr + (offset * slope)

    # Actual Beam Bottom (Input + Camber + Beam-specific vertical offset)
    beam_v_offset = (beam_elevation_offsets[i] / 12.0) if i < len(beam_elevation_offsets) else 0.0
    actual_beam_bottom = base_bot_arr + camber_arr + beam_v_offset

    # Plot Top of Deck
    plt.plot(stations_arr, top_of_deck, color=c, linestyle='-', linewidth=2, label=f'Beam {i+1} Top of Deck')
    # Plot Beam Bottom
    plt.plot(stations_arr, actual_beam_bottom, color=c, linestyle='--', linewidth=1.5, alpha=0.8, label=f'Beam {i+1} Bottom')

    # Fill the structural depth (Bottom to Deck Top)
    plt.fill_between(stations_arr, actual_beam_bottom, top_of_deck, color=c, alpha=0.1)

plt.title('Multi-Span Comparison: Top-of-Deck vs Beam-Bottom (Bottom-Up Logic)')
plt.xlabel('Station')
plt.ylabel('Elevation (ft)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 10. Total Haunch Volume Calculation
This section calculates the concrete volume of the haunch (the buildup between the top of the beam and the bottom of the deck) using numerical integration (trapezoidal rule) across the entire bridge length.

In [ ]:
# @title
import numpy as np

print("--- Calculating Total Haunch Volume ---")
# Create a high-resolution array for integration
num_pts = 1000
x_arr = np.linspace(0, bridge_length, num_pts)
stations_arr = bridge_start_sta + x_arr

# 1. Re-calculate the continuous PGL array using DRY helper
pgl_arr = get_pgl_elevation(stations_arr)

# 2. Re-calculate the Beam Bottom and Camber arrays
camber_arr = np.zeros_like(stations_arr)
base_bot_arr = np.zeros_like(stations_arr)

for _, row in spans_df.iterrows():
    mask = (stations_arr >= row['start_sta']) & (stations_arr <= row['end_sta'])
    local_x = stations_arr[mask] - row['start_sta']
    camber_arr[mask] = 4 * (row['camber_max'] / 12) * (local_x * (row['length'] - local_x)) / (row['length'] ** 2)
    base_bot_arr[mask] = row['beam_bottom_start_elev'] + (row['beam_bottom_end_elev'] - row['beam_bottom_start_elev']) * (local_x / row['length'])

# 3. Calculate Haunch Volume per Beam
total_volume_cf = 0.0
flange_width_ft = flange_width / 12.0

for i in range(num_beams):
    offset = beam_offsets_ft[i]
    beam_v_offset = (beam_elevation_offsets[i] / 12.0) if i < len(beam_elevation_offsets) else 0.0

    # Deck elevation
    slope = get_effective_cross_slope(offset)
    tod_arr = pgl_arr + (offset * slope)

    # Beam top elevation
    actual_bot_arr = base_bot_arr + camber_arr + beam_v_offset
    actual_tob_arr = actual_bot_arr + beam_depth

    # Haunch thickness (Top of Deck - Deck Thickness - Top of Beam)
    haunch_thickness_ft = tod_arr - deck_thickness - actual_tob_arr

    # Area under the thickness curve using trapezoidal integration
    beam_haunch_area_sqft = np.trapezoid(haunch_thickness_ft, x=stations_arr)
    beam_volume_cf = beam_haunch_area_sqft * flange_width_ft

    total_volume_cf += beam_volume_cf
    print(f"Beam {i+1} Haunch Volume: {beam_volume_cf:8.2f} cubic feet (approx {beam_volume_cf/27:5.2f} cubic yards)")

print("-" * 55)
print(f"Total Haunch Volume: {total_volume_cf:.2f} cubic feet")
print(f"Total Haunch Volume: {total_volume_cf / 27:.2f} cubic yards")


## 11. Haunch Volume Sanity Check
This cell visually plots the haunch thickness for each beam and compares the precise numerical integration to a simplified "ballpark" calculation:

$$\text{Ballpark Volume} = \text{Bridge Length} \times \text{Flange Width} \times \text{Average Haunch Thickness}$$

In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt

print("--- Sanity Check: Ballpark Haunch Volume ---")

total_ballpark_cf = 0.0
flange_width_ft = flange_width / 12.0

plt.figure(figsize=(12, 5))

for i in range(num_beams):
    offset = beam_offsets_ft[i]
    beam_v_offset = (beam_elevation_offsets[i] / 12.0) if i < len(beam_elevation_offsets) else 0.0

    # Deck elevation
    slope = get_effective_cross_slope(offset)
    tod_arr = pgl_arr + (offset * slope)

    # Beam top elevation
    actual_bot_arr = base_bot_arr + camber_arr + beam_v_offset
    actual_tob_arr = actual_bot_arr + beam_depth

    # Haunch thickness (ft)
    haunch_thickness_ft = tod_arr - deck_thickness - actual_tob_arr

    # 1. Calculate the simple average thickness
    avg_thick_ft = np.mean(haunch_thickness_ft)

    # 2. Ballpark volume for this beam (Rectangle: L * W * Avg_H)
    ballpark_beam_vol = bridge_length * flange_width_ft * avg_thick_ft
    total_ballpark_cf += ballpark_beam_vol

    print(f"Beam {i+1}:")
    print(f"  Avg Haunch = {avg_thick_ft * 12:4.2f} in ({avg_thick_ft:.4f} ft)")
    print(f"  Math: {bridge_length:.2f} ft (L) * {flange_width_ft:.2f} ft (W) * {avg_thick_ft:.4f} ft (Avg H) = {ballpark_beam_vol:6.2f} cf")

    # 3. Plot the thickness profile
    plt.plot(stations_arr, haunch_thickness_ft * 12, label=f'Beam {i+1} Haunch')
    plt.fill_between(stations_arr, 0, haunch_thickness_ft * 12, alpha=0.15)

print("-" * 55)
print(f"Total Ballpark Volume: {total_ballpark_cf:.2f} cubic feet")
print(f"Total Exact Volume:    {total_volume_cf:.2f} cubic feet")
difference = abs(total_ballpark_cf - total_volume_cf) / total_volume_cf * 100
print(f"Difference:            {difference:.2f}%")
print("-" * 55)

plt.title("Haunch Thickness Profile Along Bridge Length (Inches)")
plt.xlabel("Station")
plt.ylabel("Thickness (in)")
plt.axhline(haunch, color='red', linestyle='--', linewidth=1.5, label=f'Minimum Allowable Haunch ({haunch} in)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 12. Interactive Bridge Cross-Section Visualization
This tool visualizes the physical cross-section of the bridge at any given station. It renders the approximate geometry of the FIB 36 beams, the poured haunch (pedestal), and the continuous deck slab based on the longitudinal profile, cross-slope, and beam configurations.

In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.patches as patches

# --- Define FIB 36 Approximate Geometry ---
def get_fib36_polygon(center_x, bottom_y):
    # Standard FIB 36 Dimensions (in feet)
    # Depth = 36" (3.0')
    # Top Flange Width = 48" (4.0')
    # Bottom Flange Width = 38" (3.167')
    # Web Thickness = 7" (0.583')

    pts = [
        (-1.583, 0.0),    # Bottom left
        (1.583, 0.0),     # Bottom right
        (1.583, 0.583),   # Bottom flange top right
        (0.292, 0.833),   # Web bottom right
        (0.292, 2.667),   # Web top right
        (2.0, 2.667),     # Top flange bottom right
        (2.0, 3.0),       # Top right
        (-2.0, 3.0),      # Top left
        (-2.0, 2.667),    # Top flange bottom left
        (-0.292, 2.667),  # Web top left
        (-0.292, 0.833),  # Web bottom left
        (-1.583, 0.583)   # Bottom flange top left
    ]
    # Translate to location
    return [(center_x + x, bottom_y + y) for x, y in pts]

# --- Set up Interactive Plot ---
xs_slider = widgets.FloatSlider(value=bridge_start_sta + (bridge_length/2), min=bridge_start_sta, max=bridge_end_sta, step=0.01, description='Station:', layout=widgets.Layout(width='60%'))
xs_input = widgets.FloatText(value=bridge_start_sta + (bridge_length/2), description='Exact Sta:', step=0.01, layout=widgets.Layout(width='25%'))
widgets.jslink((xs_slider, 'value'), (xs_input, 'value'))
out_xs = widgets.Output()

def update_cross_section(change):
    with out_xs:
        clear_output(wait=True)
        sta = xs_slider.value

        # 1. PGL Elevation using DRY helper
        pgl_elev = get_pgl_elevation(sta)

        # 2. Span & Camber info
        current_span_df = spans_df[(spans_df['start_sta'] <= sta) & (spans_df['end_sta'] >= sta)]
        if current_span_df.empty:
            current_span = spans_df.iloc[-1] if sta > spans_df['end_sta'].max() else spans_df.iloc[0]
        else:
            current_span = current_span_df.iloc[0]

        x_span = sta - current_span['start_sta']
        s_len, s_camber = current_span['length'], current_span['camber_max']
        s_bot_start, s_bot_end = current_span['beam_bottom_start_elev'], current_span['beam_bottom_end_elev']

        camber_ft = 4 * (s_camber / 12) * (x_span * (s_len - x_span)) / (s_len ** 2)
        base_bot_ft = s_bot_start + (s_bot_end - s_bot_start) * (x_span / s_len)

        fig, ax = plt.subplots(figsize=(14, 8))

        # 3. Draw Components for each Beam
        for i in range(num_beams):
            offset = beam_offsets_ft[i]
            beam_v_offset = (beam_elevation_offsets[i] / 12.0) if i < len(beam_elevation_offsets) else 0.0
            beam_bottom_elev = base_bot_ft + camber_ft + beam_v_offset
            beam_top_elev = beam_bottom_elev + beam_depth

            # Draw FIB 36 Beam
            beam_poly = get_fib36_polygon(offset, beam_bottom_elev)
            poly_patch = patches.Polygon(beam_poly, closed=True, facecolor='gainsboro', edgecolor='black', zorder=2)
            ax.add_patch(poly_patch)

            # Calculate local deck elevations (Center, Left, Right of haunch)
            haunch_w = flange_width / 12.0
            off_L = offset - haunch_w / 2
            off_R = offset + haunch_w / 2

            tod_C = pgl_elev + offset * get_effective_cross_slope(offset, sta)
            tod_L = pgl_elev + off_L * get_effective_cross_slope(off_L, sta)
            tod_R = pgl_elev + off_R * get_effective_cross_slope(off_R, sta)

            bod_C = tod_C - deck_thickness
            bod_L = tod_L - deck_thickness
            bod_R = tod_R - deck_thickness

            # Draw Haunch (Pedestal) - Including center point (offset, bod_C) to follow the crown peak
            top_offsets = sorted(set([off_L, off_R] + ([0.0] if is_crowned and off_L < 0 < off_R else [])), reverse=True)
            haunch_poly = [(off_L, beam_top_elev), (off_R, beam_top_elev)] + [
                (x, pgl_elev + x * get_effective_cross_slope(x, sta) - deck_thickness)
                for x in top_offsets]
            haunch_patch = patches.Polygon(haunch_poly, closed=True, facecolor='burlywood', edgecolor='black', zorder=1)
            ax.add_patch(haunch_patch)

            # --- Annotations ---
            # Annotate Top of Deck (TOD)
            ax.text(offset, tod_C + 0.1, f"TOD: {tod_C:.3f}'", ha='center', va='bottom', fontsize=9, color='blue',
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))

            # Calculate Haunch Thickness
            h_thick_L_in = (bod_L - beam_top_elev) * 12
            h_thick_R_in = (bod_R - beam_top_elev) * 12

            # Group Beam Top & Haunch Annotations into a clear bounding box
            haunch_text = (f"TOB: {beam_top_elev:.3f}'\n"
                           f"Haunch: L {h_thick_L_in:.3f}\" | R {h_thick_R_in:.3f}\"")
            ax.text(offset, beam_top_elev - 0.15, haunch_text, ha='center', va='top',
                    fontsize=8, color='black',
                    bbox=dict(facecolor='white', alpha=0.9, edgecolor='purple', boxstyle='round,pad=0.3'), zorder=5)

            # Annotate beam centerlines and Bottom of Beam (BOB)
            ax.axvline(offset, color='gray', linestyle=':', alpha=0.5, zorder=0)
            ax.text(offset, beam_bottom_elev - 0.3, f'Beam {i+1}\nBOB: {beam_bottom_elev:.3f}\'' , ha='center', va='top', fontsize=9,
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))

        # 4. Draw Continuous Deck Slab
        # Assuming deck overhangs the exterior beams by 2 feet
        left_edge = beam_offsets_ft[0] - flange_width / 24.0
        right_edge = beam_offsets_ft[-1] + flange_width / 24.0

        deck_xs = np.unique(np.r_[np.linspace(left_edge, right_edge, 50), [0.0] if left_edge < 0 < right_edge else []])
        deck_tod = [pgl_elev + x * get_effective_cross_slope(x, sta) for x in deck_xs]
        deck_bod = [y - deck_thickness for y in deck_tod]

        deck_poly_x = np.concatenate([deck_xs, deck_xs[::-1]])
        deck_poly_y = np.concatenate([deck_tod, deck_bod[::-1]])
        deck_patch = patches.Polygon(list(zip(deck_poly_x, deck_poly_y)), closed=True, facecolor='lightsteelblue', edgecolor='black', zorder=3)
        ax.add_patch(deck_patch)

        # Draw PGL Marker
        ax.scatter(0, pgl_elev, color='red', marker='+', s=100, zorder=5, label='PGL')
        ax.axvline(0, color='red', linestyle='--', alpha=0.5, zorder=0)

        ax.set_aspect('equal')
        ax.set_title(f'Bridge Cross-Section @ Station {sta:.2f}')
        ax.set_xlabel('Offset from PGL (ft)')
        ax.set_ylabel('Elevation (ft)')
        ax.grid(True, linestyle='--', alpha=0.3)

        # Adjust limits to fit the drawing
        y_min = base_bot_ft + camber_ft - 1.5
        y_max = pgl_elev + 1.5
        ax.set_xlim(left_edge - 1.5, right_edge + 1.5)
        ax.set_ylim(y_min, y_max)
        ax.legend(loc='upper right')

        plt.tight_layout()
        plt.show()

xs_slider.observe(update_cross_section, names='value')
update_cross_section(None)
display(widgets.HBox([xs_slider, xs_input]), out_xs)


## 13. Pier cap and bearing-seat geometry

This section uses the **two-beam notebook geometry** to calculate pedestal tops
and the **four corners in transverse elevation**: top left (TL), top right (TR),
bottom right (BR), and bottom left (BL). These are elevation-view corners, not
the four corners of the cap's top surface in plan. Left means negative PGL offset.
All elevations use the same project datum as the existing beam-bottom model.

The interior support is at station **1700.57**, the common endpoint of the two
spans. “Midspan pier” is interpreted as this interior support, not the bridge's
arithmetic midpoint (1699.345). At a modeled support, parabolic camber is zero.
Both adjacent span ends are checked separately; no assumption that unequal
support elevations or pad thicknesses produce a common pedestal elevation is made.

### Input sources and unresolved geometry

- `Pier.dxf`: dimension block `*D5374` labels beam spacing **8 ft 7 in**;
  `*D5379` labels slab thickness **6 in**; `Beam-FI-36_21` and `_22` define
  **3 ft** deep sections. The current notebook instead uses **4.583 ft** beam
  spacing. The notebook spacing remains authoritative until that conflict is resolved.
- The revised `*D5374`–`*D5384` dimension blocks have no INSERT/DIMENSION references
  in the supplied drawing. No corresponding placed pier-cap/pedestal outline was
  located. `$INSUNITS` is 0 (unspecified). Drafting Y coordinates are **not** elevations.
- `D2_I95_Project_Journal (22).bpad`, Bridge Geometry: `CLPier_station = 1700.57`,
  `PierCap_width = 4 ft` (longitudinal width), `PierCap_ht = 3.75 ft`.
  The separate older substructure expression has `Cap_ht = 3.5 ft`.
  Neither cap depth is adopted automatically. The 4-ft width is **not** the
  transverse cap length. The journal's 36-by-36-inch pedestal is labeled an example.
- Pad thickness, adopted cap depth, transverse cap length, and a cap elevation or
  pedestal-height reference need confirmation. `None` below means **unknown**;
  it never means zero. Available results are shown while unresolved values remain blank.

The attachments are reference data; their notes are not instructions to change
the notebook's other analysis or import prior-model elevations.

### Elevation chain

At each adjacent span support and beam:

`beam_bottom = span_support_bottom + beam_vertical_offset_in / 12`

`pedestal_top = beam_bottom - bearing_pad_height_in / 12`

Choose one cap placement method:

1. Specify `minimum_pedestal_height_in` at the bearing centerlines. Then
   `cap_top_at_center = min(pedestal_top - cap_cross_slope * (beam_offset - cap_center_offset)) - minimum_pedestal_height_in / 12`.
2. Specify `cap_top_at_center_ft` directly. The resulting pedestal heights are reported.

For each end of the cap, `cap_top = cap_top_at_center + cap_cross_slope * (end_offset - cap_center_offset)`
and `cap_bottom = cap_top - cap_depth_ft`.

Changing **cap depth** holds the cap top and pedestal tops fixed and moves the
cap bottom. Changing **bearing-pad thickness** moves the pedestal top below the
fixed beam support. In minimum-pedestal-height mode the cap follows that change;
in fixed-cap-top mode only pedestal heights change.

This is a transverse projection at the idealized support station. Actual fore/aft
bearing-center offsets, skew, pedestal footprints, seat slopes, and any separate
sole plate/grout stack are not supplied and are not inferred. Pad input is the
effective vertical separation used between the beam soffit and pedestal top.
Pedestal tops are concrete bearing seats **below the beams**, separate from
the haunch concrete between beam tops and deck.


In [ ]:
# @title Pier geometry inputs — edit and rerun this section
# None = not yet established by the supplied source geometry.
pier_station_ft = None                 # Auto: unique common span support (1700.57)
bearing_pad_heights_in = None          # Inches: scalar, [Beam 1, Beam 2], or 2-by-N array
                                      # 2-by-N rows: approach-span end, departure-span start
cap_depth_ft = None                    # Vertical cap depth; journal has 3.75 and 3.5 ft
cap_length_ft = None                   # Transverse length; do not use 4-ft longitudinal width
minimum_pedestal_height_in = None      # Option A: minimum seat height above cap, at beam CL
cap_top_at_center_ft = None            # Option B: fixed project elevation; leave A as None
cap_center_offset_ft = 0.0             # Center cap on PGL (editable geometric assumption)
cap_cross_slope = 0.0                  # Flat cap top by default; ft/ft, positive rises right


In [ ]:
"""Pure geometry helpers embedded verbatim into the Jupyter notebook.

Project elevations come from span support elevations, never from DXF drawing Y.
The transverse elevation is a projection at the common support station; physical
longitudinal bearing offsets, skew and bearing compression are not modeled.
"""
import numpy as np
import pandas as pd


def _finite_number(value, name, *, positive=False, nonnegative=False):
    if isinstance(value, (bool, np.bool_)) or not np.isscalar(value):
        raise ValueError(f'{name} must be a finite number.')
    try:
        value = float(value)
    except (TypeError, ValueError):
        raise ValueError(f'{name} must be a finite number.') from None
    if not np.isfinite(value):
        raise ValueError(f'{name} must be finite.')
    if positive and value <= 0:
        raise ValueError(f'{name} must be greater than zero.')
    if nonnegative and value < 0:
        raise ValueError(f'{name} cannot be negative.')
    return value


def calculate_pier_geometry(spans, beam_offsets_ft, beam_raises_in, *,
                            pier_station_ft=None, bearing_pad_heights_in=None,
                            cap_depth_ft=None, cap_length_ft=None,
                            minimum_pedestal_height_in=None,
                            cap_top_at_center_ft=None, cap_center_offset_ft=0.0,
                            cap_cross_slope=0.0):
    """Return support, pedestal and four elevation-corner tables.

    Pad heights are vertical effective thicknesses: a scalar, one value per
    beam, or a 2-by-N array [approach span end, departure span start].
    Cap top is a transverse straight line. Depth is vertical and constant.
    The minimum pedestal-height option sets the smallest centerline pedestal
    height among all support seats; it is a geometric input, not a code minimum.
    """
    required = ['span_id', 'start_sta', 'end_sta',
                'beam_bottom_start_elev', 'beam_bottom_end_elev']
    if any(c not in spans for c in required) or len(spans) < 2:
        raise ValueError('Pier geometry requires at least two spans with support elevations.')
    ordered = spans.sort_values('start_sta').reset_index(drop=True)
    for name in required[1:]:
        if not np.all(np.isfinite(ordered[name].to_numpy(dtype=float))):
            raise ValueError(f'Span {name} must be finite.')
    if np.any(ordered['end_sta'] <= ordered['start_sta']):
        raise ValueError('Every span must have a positive length.')
    end_stations = ordered['end_sta'].to_numpy(dtype=float)[:-1]
    start_stations = ordered['start_sta'].to_numpy(dtype=float)[1:]
    if not np.all(np.isclose(end_stations, start_stations, rtol=0, atol=1e-8)):
        raise ValueError('Spans must be contiguous without gaps or overlaps.')
    if pier_station_ft is None:
        if len(end_stations) != 1:
            raise ValueError('Set pier_station_ft explicitly when there are multiple interior supports.')
        station = float(end_stations[0])
    else:
        station = _finite_number(pier_station_ft, 'pier_station_ft')
    matches = np.flatnonzero(np.isclose(end_stations, station, rtol=0, atol=1e-8))
    if len(matches) != 1:
        raise ValueError('Pier station must coincide with an existing interior span support.')
    j = int(matches[0])
    adjacent = [ordered.iloc[j], ordered.iloc[j + 1]]

    offsets = np.asarray(beam_offsets_ft, dtype=float)
    raises = np.asarray(beam_raises_in, dtype=float)
    if offsets.ndim != 1 or not len(offsets) or not np.all(np.isfinite(offsets)):
        raise ValueError('Beam offsets must be a nonempty finite vector.')
    if np.any(np.diff(offsets) <= 0):
        raise ValueError('Beam offsets must be strictly increasing.')
    if raises.ndim == 0:
        raises = np.full(len(offsets), float(raises))
    if raises.shape != offsets.shape or not np.all(np.isfinite(raises)):
        raise ValueError('Provide one finite vertical offset per beam, or a scalar.')

    center = _finite_number(cap_center_offset_ft, 'cap_center_offset_ft')
    slope = _finite_number(cap_cross_slope, 'cap_cross_slope')
    if cap_depth_ft is not None:
        cap_depth_ft = _finite_number(cap_depth_ft, 'cap_depth_ft', positive=True)
    if cap_length_ft is not None:
        cap_length_ft = _finite_number(cap_length_ft, 'cap_length_ft', positive=True)
        if np.any(np.abs(offsets - center) >= cap_length_ft / 2):
            raise ValueError('All beam centerlines must lie strictly inside the cap ends.')
    if minimum_pedestal_height_in is not None:
        minimum_pedestal_height_in = _finite_number(
            minimum_pedestal_height_in, 'minimum_pedestal_height_in', nonnegative=True)
    if cap_top_at_center_ft is not None:
        cap_top_at_center_ft = _finite_number(cap_top_at_center_ft, 'cap_top_at_center_ft')
    if minimum_pedestal_height_in is not None and cap_top_at_center_ft is not None:
        raise ValueError('Choose minimum pedestal height OR fixed cap-top elevation, not both.')

    if bearing_pad_heights_in is None:
        pads = np.full((2, len(offsets)), np.nan)
    else:
        pads = np.asarray(bearing_pad_heights_in, dtype=float)
        if pads.ndim == 0:
            pads = np.full((2, len(offsets)), float(pads))
        elif pads.shape == offsets.shape:
            pads = np.tile(pads, (2, 1))
        if pads.shape != (2, len(offsets)) or not np.all(np.isfinite(pads)) or np.any(pads <= 0):
            raise ValueError('Pad heights must be positive finite inches: scalar, N values, or 2-by-N values.')

    rows = []
    for side, span in enumerate(adjacent):
        endpoint = 'end' if side == 0 else 'start'
        base = float(span[f'beam_bottom_{endpoint}_elev'])
        # Camber is zero at a span support; never use midspan camber here.
        for b, (offset, raise_in) in enumerate(zip(offsets, raises)):
            bottom = base + raise_in / 12.0
            pad = pads[side, b]
            rows.append({
                'Support': 'Approach span end' if side == 0 else 'Departure span start',
                'Span': span['span_id'], 'Beam': b + 1, 'Station (ft)': station,
                'Offset from PGL (ft)': offset, 'Beam bottom / pad top (ft)': bottom,
                'Bearing pad height (in)': pad, 'Pedestal top (ft)': bottom - pad / 12.0,
            })
    seats = pd.DataFrame(rows)
    missing = []
    if bearing_pad_heights_in is None:
        missing.append('bearing_pad_heights_in: effective vertical bearing-pad thickness')

    cap_top = cap_top_at_center_ft
    if cap_top is None and minimum_pedestal_height_in is not None and bearing_pad_heights_in is not None:
        cap_top = float(np.min(seats['Pedestal top (ft)'].to_numpy()
                              - slope * (seats['Offset from PGL (ft)'].to_numpy() - center))
                        - minimum_pedestal_height_in / 12.0)
    if minimum_pedestal_height_in is None and cap_top_at_center_ft is None:
        missing.append('minimum_pedestal_height_in OR cap_top_at_center_ft: cap vertical placement')
    if cap_depth_ft is None:
        missing.append('cap_depth_ft: vertical top-to-bottom cap depth')
    if cap_length_ft is None:
        missing.append('cap_length_ft: transverse end-to-end cap length')

    seats['Cap top at beam CL (ft)'] = (np.nan if cap_top is None else
                                      cap_top + slope * (seats['Offset from PGL (ft)'] - center))
    seats['Pedestal height at CL (in)'] = (
        seats['Pedestal top (ft)'] - seats['Cap top at beam CL (ft)']) * 12.0
    if np.any(seats['Pedestal height at CL (in)'].to_numpy() < -1e-8):
        raise ValueError('Cap top is above a pedestal top; check cap placement and pad heights.')

    corners = []
    for name, side, is_bottom in [('TL — top left', -1, False), ('TR — top right', 1, False),
                                   ('BR — bottom right', 1, True), ('BL — bottom left', -1, True)]:
        x = np.nan if cap_length_ft is None else center + side * cap_length_ft / 2
        z = np.nan
        if cap_top is not None and (slope == 0 or np.isfinite(x)):
            z = cap_top if slope == 0 else cap_top + slope * (x - center)
            if is_bottom:
                z = np.nan if cap_depth_ft is None else z - cap_depth_ft
        corners.append({'Corner in transverse elevation': name,
                        'Station (ft)': station, 'Offset from PGL (ft)': x, 'Elevation (ft)': z})
    return {'station_ft': station, 'seats': seats, 'corners': pd.DataFrame(corners),
            'cap_top_at_center_ft': cap_top, 'missing_inputs': missing,
            'cap_center_offset_ft': center, 'cap_cross_slope': slope,
            'cap_depth_ft': cap_depth_ft, 'cap_length_ft': cap_length_ft}


def plot_pier_geometry(result, beam_depth_ft):
    """Transverse elevation; seat rectangles are symbols, not pedestal footprints."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Polygon, Rectangle
    beam_depth_ft = _finite_number(beam_depth_ft, 'beam_depth_ft', positive=True)
    corners, seats = result['corners'], result['seats']
    if result['missing_inputs'] or not np.all(np.isfinite(corners['Elevation (ft)'])):
        print('Pier elevation drawing awaits the missing geometry inputs listed above.')
        return None
    fig, ax = plt.subplots(figsize=(12, 7))
    polygon = corners[['Offset from PGL (ft)', 'Elevation (ft)']].to_numpy()
    ax.add_patch(Polygon(polygon, closed=True, facecolor='#dbe3eb', edgecolor='#344556'))
    for _, row in corners.iterrows():
        x, z = row['Offset from PGL (ft)'], row['Elevation (ft)']
        ax.scatter(x, z, s=24, color='#344556')
        left = 'left' in row['Corner in transverse elevation']
        ax.annotate(f"{row['Corner in transverse elevation'].split(' — ')[0]}  {z:.4f} ft",
                    (x, z), xytext=(-10 if left else 10, 10 if 'top' in row.iloc[0] else -18),
                    textcoords='offset points', ha='right' if left else 'left', fontsize=9)
    groups = seats.groupby('Support', sort=False)
    colors = ['#ba853d', '#4384a0']
    for side_index, (support, group) in enumerate(groups):
        for _, row in group.iterrows():
            x = row['Offset from PGL (ft)']
            z = row['Pedestal top (ft)']
            bottom = row['Beam bottom / pad top (ft)']
            cap = row['Cap top at beam CL (ft)']
            # Separate the two support-face symbols slightly in the projection.
            dx = -0.17 if side_index == 0 else 0.17
            ax.plot([x + dx, x + dx], [cap, z], color=colors[side_index], lw=6)
            ax.add_patch(Rectangle((x + dx - .10, z), .20, bottom-z,
                                   facecolor='#333333', edgecolor='black'))
            ax.plot([x-.35, x+.35], [bottom, bottom], color='#344556', lw=2)
            ax.plot([x, x], [bottom, bottom+min(beam_depth_ft, .7)], color='#344556', ls=':')
        ax.plot([], [], color=colors[side_index], lw=6, label=support)
    for beam, group in seats.groupby('Beam', sort=False):
        x = float(group.iloc[0]['Offset from PGL (ft)'])
        levels = group['Pedestal top (ft)'].to_numpy()
        level_text = f'{levels[0]:.4f}' if np.allclose(levels, levels[0], rtol=0, atol=1e-8) else ' / '.join(f'{v:.4f}' for v in levels)
        ax.text(x, group['Beam bottom / pad top (ft)'].max()+.9,
                f'Beam {int(beam)}\nPedestal top {level_text} ft', ha='center', fontsize=9)
    ax.axvline(0, color='gray', ls=':', lw=.8)
    length = result['cap_length_ft']
    ax.set_xlim(result['cap_center_offset_ft']-length/2-2,
                result['cap_center_offset_ft']+length/2+2)
    ax.set_ylim(corners['Elevation (ft)'].min()-.8,
                seats['Beam bottom / pad top (ft)'].max()+1.7)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('Transverse offset from PGL (ft); left = negative')
    ax.set_ylabel('Project elevation (ft)')
    ax.set_title(f"Pier at station {result['station_ft']:.2f} — transverse elevation\n"
                 'Pedestal/bearing symbols offset for readability; widths are schematic')
    ax.grid(alpha=.15); ax.legend(loc='lower center', bbox_to_anchor=(.5, -.28), ncol=2)
    fig.tight_layout()
    return fig


In [ ]:
# @title Pier elevations and transverse elevation drawing
pier_results = calculate_pier_geometry(
    spans_df, beam_offsets_ft, beam_elevation_offsets,
    pier_station_ft=pier_station_ft,
    bearing_pad_heights_in=bearing_pad_heights_in,
    cap_depth_ft=cap_depth_ft, cap_length_ft=cap_length_ft,
    minimum_pedestal_height_in=minimum_pedestal_height_in,
    cap_top_at_center_ft=cap_top_at_center_ft,
    cap_center_offset_ft=cap_center_offset_ft, cap_cross_slope=cap_cross_slope)

pier_seat_elevations = pier_results['seats']
pier_cap_corner_elevations = pier_results['corners']
print(f"Interior pier support station: {pier_results['station_ft']:.3f} ft")
print(f"Active beam offsets from notebook (ft): {np.asarray(beam_offsets_ft)}")
if len(beam_offsets_ft) == 2 and not np.isclose(np.diff(beam_offsets_ft)[0], 8 + 7/12, rtol=0, atol=1e-6):
    print('SOURCE DIFFERENCE: notebook beam spacing differs from DXF/journal 8 ft 7 in.')
if pier_results['missing_inputs']:
    print('INCOMPLETE GEOMETRY — fill these inputs; blank elevations are not zero:')
    for missing in pier_results['missing_inputs']:
        print(' - ' + missing)
print('Bearing-seat elevations: separate rows for both adjacent span ends.')
display(pier_seat_elevations.round(4).astype(object).where(pier_seat_elevations.notna(), 'NOT SET'))
print('Four pier-cap corners in transverse elevation:')
display(pier_cap_corner_elevations.round(4).astype(object).where(pier_cap_corner_elevations.notna(), 'NOT SET'))
pier_figure = plot_pier_geometry(pier_results, beam_depth)
if pier_figure is not None:
    plt.show()


## 14. Data-Driven Mind Map
Using `graphviz`, we can recursively generate a mind map from any hierarchical data structure (like nested dictionaries or lists). When your underlying data changes, simply re-run the cell to update the visualization.

In [ ]:
import graphviz
from IPython.display import display

# 1. Define your data (this can be loaded dynamically from an API, DataFrame, or JSON)
mind_map_data = {
    "Bridge Design Project": {
        "Global Parameters": ["Span Length", "Beam Spacing", "Deck Thickness"],
        "Analysis": {
            "Geometry": ["Vertical Curve", "Cross-Slope"],
            "Clearance": ["Haunch Calculation", "Roadway Envelope"]
        },
        "Deliverables": ["Total Volumes", "Elevation Reports", "Cross-Section Plots"]
    }
}

# 2. Recursive function to build the mind map
def create_mind_map(data, graph=None, parent=None, is_root=True):
    if graph is None:
        # Initialize graph with visual styling
        graph = graphviz.Digraph(engine='dot')
        graph.attr(rankdir='LR', splines='spline', nodesep='0.4', ranksep='0.7') # Left to Right layout
        graph.attr('edge', color='#546e7a', penwidth='1.5')

    if isinstance(data, dict):
        for key, value in data.items():
            node_id = str(key)

            # Style the root node differently
            if is_root:
                graph.node(node_id, label=str(key), shape='box', style='rounded,filled', fillcolor='#ffcc80', fontname='Arial', fontsize='14', fontcolor='black')
            else:
                graph.node(node_id, label=str(key), shape='box', style='rounded,filled', fillcolor='#e1f5fe', fontname='Arial', fontcolor='black')

            if parent is not None:
                graph.edge(str(parent), node_id)

            # Recurse for children
            create_mind_map(value, graph, node_id, is_root=False)

    elif isinstance(data, list):
        for item in data:
            node_id = str(item)
            # Style for leaf nodes (endpoints)
            graph.node(node_id, label=str(item), shape='box', style='rounded,filled', fillcolor='#c8e6c9', fontname='Arial', fontcolor='black')
            if parent is not None:
                graph.edge(str(parent), node_id)

    return graph

# 3. Generate and display the map
mind_map = create_mind_map(mind_map_data)
display(mind_map)